# Static export: one file, no backend

`m.save("map.html")` writes the whole map — layer configs, coordinate/time/style
buffers, the widget bundle and CSS — into one self-contained HTML file that opens
from disk or a static host with no Python behind it. The sidebar and time playback
work fully client-side.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from swiftmap import Map

rng = np.random.default_rng(6)
n = 400
df = pd.DataFrame({
    "lat": 36.03 + rng.normal(0, 0.05, n),
    "lon": -5.45 + rng.normal(0, 0.09, n),
    "reading": np.round(rng.gamma(4, 4, n), 1),
    "timestamp": pd.date_range("2026-08-01", periods=n, freq="15min", tz="UTC"),
})

m = Map()
m.add_circle_markers(df, name="Sensors", color_col="reading",
                     layer_group="Feeds")
m.make_time_layer("Sensors", period="PT6H", duration=None)
m.save("export_demo.html")

kb = Path("export_demo.html").stat().st_size / 1024
print(f"export_demo.html: {kb:,.0f} KB")

Open `export_demo.html` from disk: the sidebar toggles, the time slider plays, the
view opens framed on the data (the automatic fit rides along).

Worth knowing:

- Buffers ship base64-encoded — expect roughly **4/3 of their binary size**.
- Leaflet and glify load from unpkg **when the file is viewed**, same as the live
  widget — so viewing needs internet even though nothing else does.
- Interaction stays in the file: clicks and toggles have no Python to sync back
  to. The baked state is reachable from the browser console as `window.__model`
  for debugging.

## `to_html()` — the same document as a string

Which is also the Streamlit story:

```python
import streamlit.components.v1 as components
components.html(m.to_html(), height=600)
```

In [ ]:
html = m.to_html(title="Sensor sweep")
print(f"{len(html) / 1024:,.0f} KB of HTML, starts: {html[:60]!r}")

For live bidirectional apps — selections driving the map, clicks driving Python —
that's Shiny territory: see `shiny/01_basic_app.py` and
`shiny/02_linked_table.py`.